# Nool — Contriever Passage Embedding Generation (Google Colab)

Welcome to the **Nool** GPU embedding generation workflow. This notebook encodes
the 50,000-passage HotpotQA benchmark sample into dense vector representations
using Facebook's original unsupervised **Contriever** model (`facebook/contriever`).

---

## Quick Start Instructions

1. **Open in Google Colab**:
   - [Open in Colab](https://colab.research.google.com/github/nayanojwal0810/Nool/blob/main/notebooks/contriever_embeddings_colab.ipynb)
2. **Enable GPU Hardware Acceleration**:
   - In the Colab menu, select **Runtime** → **Change runtime type**.
   - Under **Hardware accelerator**, select **T4 GPU** (or any available GPU: V100, A100).
   - Click **Save**.
3. **Upload the Prepared Corpus to Google Drive**:
   - In your Google Drive, create the folder path:
     `MyDrive/Nool/input/`
   - Upload your local file from your Nool project:
     `data/processed/hotpotqa_sample/corpus.jsonl`
     so it resides at:
     `/content/drive/MyDrive/Nool/input/corpus.jsonl`
4. **Run the Notebook**:
   - Run each cell sequentially from top to bottom (**Runtime** → **Run all**).
   - Checkpoints are saved to Google Drive every 2,500 passages, so you can safely resume if disconnected.
5. **Download the Final Artifact**:
   - When complete, download `contriever_hotpotqa_sample.zip` from your Google Drive:
     `MyDrive/Nool/contriever_runs/hotpotqa_sample/contriever_hotpotqa_sample.zip`
   - Extract it into your local project at:
     `indexes/contriever_hotpotqa_sample/`

---

## 1. Environment & GPU Verification

This section checks your Python version, PyTorch version, and verifies that a
CUDA-capable GPU is available in your Colab session. If a GPU is not enabled,
the cell raises an error prompting you to enable one.

In [ ]:
import sys

import torch

print(f"Python Version:  {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is not enabled!\n"
        "Please switch to a GPU runtime before proceeding:\n"
        "  1. Go to top menu: Runtime -> Change runtime type\n"
        "  2. Under Hardware accelerator, select 'T4 GPU'\n"
        "  3. Click Save and re-run this cell."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"GPU Device:      {gpu_name} ({gpu_mem:.2f} GB total memory)")
print("\nGPU environment verified successfully!")

## 2. Mount Google Drive

We mount your Google Drive so that input data is read from persistent storage
and all checkpoint files and final artifacts are safely preserved even if the
Colab session disconnects.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DRIVE_NOOL_ROOT = Path("/content/drive/MyDrive/Nool")
INPUT_DIR = DRIVE_NOOL_ROOT / "input"
RUN_DIR = DRIVE_NOOL_ROOT / "contriever_runs" / "hotpotqa_sample"
CHECKPOINTS_DIR = RUN_DIR / "checkpoints"
ARTIFACT_DIR = RUN_DIR / "artifact"

INPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Google Drive mounted and directories initialized:")
print(f" - Input Directory:       {INPUT_DIR}")
print(f" - Checkpoints Directory: {CHECKPOINTS_DIR}")
print(f" - Artifact Directory:    {ARTIFACT_DIR}")

## 3. Clone Nool Repository

We clone the official Nool repository to access project utilities and record
the exact commit SHA for provenance. We do not install the optional `[dense]`
extra on Colab to prevent modifying Colab's native GPU PyTorch environment.

In [ ]:
import subprocess

REPO_URL = "https://github.com/nayanojwal0810/Nool.git"
LOCAL_REPO_DIR = Path("/content/Nool")

if not LOCAL_REPO_DIR.exists():
    print(f"Cloning Nool repository from {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, str(LOCAL_REPO_DIR)], check=True)
else:
    print("Pulling latest Nool updates...")
    subprocess.run(["git", "-C", str(LOCAL_REPO_DIR), "pull"], check=True)

commit_proc = subprocess.run(
    ["git", "-C", str(LOCAL_REPO_DIR), "rev-parse", "HEAD"],
    capture_output=True,
    text=True,
    check=True,
)
GIT_COMMIT_SHA = commit_proc.stdout.strip()
print(f"Active Nool Commit SHA: {GIT_COMMIT_SHA}")

src_path = str(LOCAL_REPO_DIR / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

## 4. Install Dependencies

We install the pinned `transformers==5.22.1` library. Notice we do not
reinstall PyTorch, preserving Colab's native CUDA-enabled build.

In [ ]:
!pip install --quiet transformers==5.22.1

import transformers

print(f"Transformers Version: {transformers.__version__}")
assert torch.cuda.is_available(), (
    "CUDA must remain available after dependency installation."
)
print("Dependencies verified successfully!")

## 5. Validate Input Corpus

We read `/content/drive/MyDrive/Nool/input/corpus.jsonl` and verify:
1. The file exists and is readable.
2. It contains exactly 50,000 passages.
3. Every record has a unique, non-empty `_id` and non-empty `text`.
4. The SHA-256 matches the expected checksum from Nool's HotpotQA sample.

In [ ]:
import hashlib
import json

CORPUS_FILE = INPUT_DIR / "corpus.jsonl"
EXPECTED_SAMPLE_SHA256 = (
    "c3970d91ebfd36d794066b42bd7acb85a744aabba2c97dd9a9bfde52d3dcbfbf"
)
EXPECTED_PASSAGE_COUNT = 50000

if not CORPUS_FILE.exists():
    raise FileNotFoundError(
        f"Corpus file not found at: {CORPUS_FILE}\n"
        "Please upload 'corpus.jsonl' from your local project's "
        "'data/processed/hotpotqa_sample/corpus.jsonl' to Drive at:\n"
        f"{CORPUS_FILE}"
    )

print(f"Computing SHA-256 for: {CORPUS_FILE}...")
hasher = hashlib.sha256()
with open(CORPUS_FILE, "rb") as f:
    while chunk := f.read(1024 * 1024):
        hasher.update(chunk)
CORPUS_SHA256 = hasher.hexdigest()
print(f"Corpus SHA-256: {CORPUS_SHA256}")

if CORPUS_SHA256 != EXPECTED_SAMPLE_SHA256:
    print("WARNING: Corpus checksum differs from standard HotpotQA sample.")
    print(f"  Expected: {EXPECTED_SAMPLE_SHA256}")
    print(f"  Actual:   {CORPUS_SHA256}")
else:
    print("Corpus SHA-256 matches expected HotpotQA sample exactly!")

passages = []
seen_ids = set()
with open(CORPUS_FILE, encoding="utf-8") as f:
    for line_no, raw_line in enumerate(f, 1):
        line = raw_line.strip()
        if not line:
            continue
        record = json.loads(line)
        doc_id = str(record.get("_id", "")).strip()
        if not doc_id:
            raise ValueError(f"Empty document ID at line {line_no}")
        if doc_id in seen_ids:
            raise ValueError(f"Duplicate document ID '{doc_id}'")
        seen_ids.add(doc_id)

        title = str(record.get("title", "")).strip() if record.get("title") else ""
        text = str(record.get("text", "")).strip()
        if not text:
            raise ValueError(f"Missing text for doc '{doc_id}'")

        search_text = f"{title} {text}".strip()
        passages.append(
            {
                "_id": doc_id,
                "search_text": search_text,
            }
        )

print(f"Validated {len(passages):,} passages in corpus (unique IDs: {len(seen_ids):,})")
if len(passages) != EXPECTED_PASSAGE_COUNT:
    print(
        f"Notice: Passage count is {len(passages)}, expected {EXPECTED_PASSAGE_COUNT}."
    )

## 6. Load Contriever Model

We load the official unsupervised `facebook/contriever` model pinned to the
verified immutable revision SHA `2bd46a25019aeea091fd42d1f0fd4801675cf699`.
We place it in evaluation mode on CUDA and run a smoke test verifying
masked mean pooling.

In [ ]:
import numpy as np
from transformers import AutoModel, AutoTokenizer

MODEL_ID = "facebook/contriever"
MODEL_REVISION = "2bd46a25019aeea091fd42d1f0fd4801675cf699"
MAX_SEQ_LENGTH = 512
EMBEDDING_DIM = 768

print(f"Loading tokenizer for {MODEL_ID} (rev: {MODEL_REVISION[:10]}...)...\n")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)

print(f"Loading model weights for {MODEL_ID}...")
model = AutoModel.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model.eval()

device = torch.device("cuda")
model.to(device)

print(f"Model loaded successfully on {torch.cuda.get_device_name(0)}!")
print(f" - Hidden Dimension:       {model.config.hidden_size}")
print(f" - Max Position Sequence:  {model.config.max_position_embeddings}")


def mean_pooling(token_embeddings, attention_mask):
    token_embeddings = token_embeddings.masked_fill(
        ~attention_mask[..., None].bool(), 0.0
    )
    mask_sum = attention_mask.sum(dim=1, keepdim=True).clamp(min=1e-9)
    return token_embeddings.sum(dim=1) / mask_sum


sample_texts = [
    "Where was Marie Curie born?",
    "Maria Sklodowska, later known as Marie Curie, was born in Warsaw.",
]
smoke_inputs = tokenizer(
    sample_texts,
    padding=True,
    truncation=True,
    max_length=MAX_SEQ_LENGTH,
    return_tensors="pt",
).to(device)

with torch.no_grad():
    smoke_outputs = model(**smoke_inputs)
    smoke_emb = mean_pooling(smoke_outputs[0], smoke_inputs["attention_mask"])

assert smoke_emb.shape == (2, EMBEDDING_DIM), f"Unexpected shape: {smoke_emb.shape}"
assert torch.isfinite(smoke_emb).all(), "Smoke test produced non-finite values!"
print(f"Smoke test PASSED! Shape: {smoke_emb.shape}, dtype: {smoke_emb.dtype}")

## 7. Generate Passage Embeddings (with Checkpointing & Resume)

### Checkpoint and Recovery Architecture
- **Chunk size**: 2,500 passages per checkpoint shard (`shard_00000.npz` to `shard_00019.npz`).
- **Initial batch size**: 16 passages per GPU forward pass.
- **OOM Recovery**: If a batch triggers CUDA out-of-memory, the GPU cache is emptied,
  the batch size is halved, and the batch is retried safely.
- **Resume capability**: If the notebook is restarted or reconnected, completed valid
  shards are detected and reused immediately.

In [ ]:
import time

CHUNK_SIZE = 2500
TOTAL_PASSAGES = len(passages)
NUM_CHUNKS = (TOTAL_PASSAGES + CHUNK_SIZE - 1) // CHUNK_SIZE
current_batch_size = 16
MIN_BATCH_SIZE = 1

print(f"Total passages:     {TOTAL_PASSAGES:,}")
print(f"Chunk size:         {CHUNK_SIZE:,} passages")
print(f"Total shards:       {NUM_CHUNKS}")
print(f"Initial batch size: {current_batch_size}")
print(f"Checkpoints dir:    {CHECKPOINTS_DIR}\n")

start_all_time = time.time()

for chunk_idx in range(NUM_CHUNKS):
    start_idx = chunk_idx * CHUNK_SIZE
    end_idx = min(start_idx + CHUNK_SIZE, TOTAL_PASSAGES)
    shard_file = CHECKPOINTS_DIR / f"shard_{chunk_idx:05d}.npz"
    tmp_shard_file = CHECKPOINTS_DIR / f"shard_{chunk_idx:05d}.tmp.npz"

    expected_chunk_ids = [p["_id"] for p in passages[start_idx:end_idx]]
    expected_chunk_count = len(expected_chunk_ids)

    if shard_file.exists():
        try:
            data = np.load(str(shard_file))
            cached_embeddings = data["embeddings"]
            cached_doc_ids = data["doc_ids"].tolist()
            cached_sha = str(data["corpus_sha256"])

            if (
                cached_embeddings.shape == (expected_chunk_count, EMBEDDING_DIM)
                and cached_doc_ids == expected_chunk_ids
                and cached_sha == CORPUS_SHA256
                and np.isfinite(cached_embeddings).all()
            ):
                print(
                    f"[{chunk_idx + 1:02d}/{NUM_CHUNKS}] Shard {chunk_idx:05d} "
                    f"valid on Drive ({expected_chunk_count:,} passages). Reusing."
                )
                continue
            else:
                print(
                    f"[{chunk_idx + 1:02d}/{NUM_CHUNKS}] Shard {chunk_idx:05d} "
                    "mismatched. Recomputing..."
                )
        except Exception as exc:
            print(
                f"[{chunk_idx + 1:02d}/{NUM_CHUNKS}] Shard {chunk_idx:05d} "
                f"read error: {exc}. Recomputing..."
            )

    print(
        f"\n[{chunk_idx + 1:02d}/{NUM_CHUNKS}] Encoding chunk {chunk_idx:05d} "
        f"({start_idx:,} to {end_idx:,})..."
    )
    chunk_texts = [p["search_text"] for p in passages[start_idx:end_idx]]
    chunk_embeddings = []

    chunk_offset = 0
    chunk_len = len(chunk_texts)
    chunk_start_time = time.time()

    while chunk_offset < chunk_len:
        batch_end = min(chunk_offset + current_batch_size, chunk_len)
        batch_texts = chunk_texts[chunk_offset:batch_end]

        try:
            inputs = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=MAX_SEQ_LENGTH,
                return_tensors="pt",
            ).to(device)

            with torch.inference_mode():
                outputs = model(**inputs)
                pooled = mean_pooling(outputs[0], inputs["attention_mask"])

            batch_np = pooled.cpu().to(torch.float32).numpy()
            if not np.isfinite(batch_np).all():
                raise ValueError(
                    f"Non-finite values in chunk {chunk_idx} offset {chunk_offset}"
                )

            chunk_embeddings.append(batch_np)
            chunk_offset = batch_end

            if chunk_offset % 500 == 0 or chunk_offset == chunk_len:
                total_done = start_idx + chunk_offset
                pct = (total_done / TOTAL_PASSAGES) * 100
                print(
                    f"  -> Done {total_done:,}/{TOTAL_PASSAGES:,} "
                    f"({pct:.1f}%) | Batch: {current_batch_size}"
                )

        except (torch.cuda.OutOfMemoryError, RuntimeError) as exc:
            is_oom = "out of memory" in str(exc).lower() or isinstance(
                exc, torch.cuda.OutOfMemoryError
            )
            if is_oom:
                torch.cuda.empty_cache()
                old_bs = current_batch_size
                current_batch_size //= 2
                print(
                    f"\n  [CUDA OOM] Reducing batch size: "
                    f"{old_bs} -> {current_batch_size}..."
                )
                if current_batch_size < MIN_BATCH_SIZE:
                    raise RuntimeError("CUDA OOM even with batch_size=1!") from exc
            else:
                raise

    chunk_matrix = np.vstack(chunk_embeddings).astype(np.float32)
    assert chunk_matrix.shape == (expected_chunk_count, EMBEDDING_DIM)

    np.savez_compressed(
        str(tmp_shard_file),
        embeddings=chunk_matrix,
        doc_ids=np.array(expected_chunk_ids, dtype=object),
        start_idx=start_idx,
        end_idx=end_idx,
        corpus_sha256=CORPUS_SHA256,
        model_id=MODEL_ID,
        model_revision=MODEL_REVISION,
        pooling="masked_mean",
        max_seq_length=MAX_SEQ_LENGTH,
        output_dtype="float32",
    )
    tmp_shard_file.replace(shard_file)
    chunk_elapsed = time.time() - chunk_start_time
    print(f"  [Saved] Shard {chunk_idx:05d} ({chunk_elapsed:.1f}s): {shard_file.name}")

total_elapsed = time.time() - start_all_time
print(f"\nAll {NUM_CHUNKS} shards completed in {total_elapsed / 60:.1f} minutes!")

## 8. Combine Shards, Validate Artifact & Create ZIP

This section:
1. Loads all checkpoint shards in sequence from Google Drive.
2. Assembles the final 50,000 × 768 float32 embedding matrix.
3. Validates order, alignment, dimensions, and finite values.
4. Writes `passage_embeddings.npy`, `document_ids.json`, and `index_manifest.json`.
5. Packages the 3 files into `contriever_hotpotqa_sample.zip` in your Google Drive.

In [ ]:
import datetime
import zipfile

print("Combining checkpoint shards from Drive...")
all_shard_matrices = []
all_shard_ids = []

for chunk_idx in range(NUM_CHUNKS):
    shard_file = CHECKPOINTS_DIR / f"shard_{chunk_idx:05d}.npz"
    if not shard_file.exists():
        raise FileNotFoundError(f"Missing shard {shard_file}! Check previous cell.")
    data = np.load(str(shard_file))
    all_shard_matrices.append(data["embeddings"])
    all_shard_ids.extend(data["doc_ids"].tolist())

final_embeddings = np.vstack(all_shard_matrices).astype(np.float32)
print(
    f"Final embedding matrix shape: {final_embeddings.shape}, "
    f"dtype: {final_embeddings.dtype}"
)
print(f"Total document IDs:          {len(all_shard_ids):,}")

assert final_embeddings.shape == (TOTAL_PASSAGES, EMBEDDING_DIM), (
    f"Shape mismatch: {final_embeddings.shape}"
)
assert len(all_shard_ids) == TOTAL_PASSAGES, f"ID count mismatch: {len(all_shard_ids)}"
assert np.isfinite(final_embeddings).all(), (
    "Final embeddings contain non-finite values!"
)

expected_all_ids = [p["_id"] for p in passages]
assert all_shard_ids == expected_all_ids, "Document IDs alignment mismatch with corpus!"

final_emb_file = ARTIFACT_DIR / "passage_embeddings.npy"
np.save(str(final_emb_file), final_embeddings)
print(f"Saved: {final_emb_file}")

emb_hasher = hashlib.sha256()
with open(final_emb_file, "rb") as f:
    while chunk := f.read(1024 * 1024):
        emb_hasher.update(chunk)
EMBEDDING_SHA256 = emb_hasher.hexdigest()
print(f"Embeddings SHA-256: {EMBEDDING_SHA256}")

final_ids_file = ARTIFACT_DIR / "document_ids.json"
with open(final_ids_file, "w", encoding="utf-8") as f:
    json.dump(all_shard_ids, f, indent=2)
print(f"Saved: {final_ids_file}")

ids_hasher = hashlib.sha256()
with open(final_ids_file, "rb") as f:
    while chunk := f.read(1024 * 1024):
        ids_hasher.update(chunk)
DOC_IDS_SHA256 = ids_hasher.hexdigest()
print(f"Document IDs SHA-256: {DOC_IDS_SHA256}")

manifest = {
    "artifact_schema_version": "1.0.0",
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "embedding_dimension": EMBEDDING_DIM,
    "pooling": "masked_mean",
    "max_token_length": MAX_SEQ_LENGTH,
    "normalize_embeddings": False,
    "similarity_method": "dot_product",
    "output_dtype": "float32",
    "passage_count": TOTAL_PASSAGES,
    "source_corpus_filename": "corpus.jsonl",
    "source_corpus_sha256": CORPUS_SHA256,
    "document_ids_file": "document_ids.json",
    "document_ids_sha256": DOC_IDS_SHA256,
    "embedding_file": "passage_embeddings.npy",
    "embedding_sha256": EMBEDDING_SHA256,
    "pytorch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "numpy_version": np.__version__,
    "gpu_name": torch.cuda.get_device_name(0),
    "inference_precision": "float32",
    "batch_size_used": current_batch_size,
    "created_at": datetime.datetime.now(datetime.UTC).isoformat(),
    "git_commit_sha": GIT_COMMIT_SHA,
}

manifest_file = ARTIFACT_DIR / "index_manifest.json"
with open(manifest_file, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)
print(f"Saved: {manifest_file}")

ZIP_DEST = RUN_DIR / "contriever_hotpotqa_sample.zip"
print(f"\nCreating ZIP archive: {ZIP_DEST}...")
with zipfile.ZipFile(ZIP_DEST, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(final_emb_file, arcname="passage_embeddings.npy")
    zf.write(final_ids_file, arcname="document_ids.json")
    zf.write(manifest_file, arcname="index_manifest.json")

zip_size_mb = ZIP_DEST.stat().st_size / (1024 * 1024)
print("=" * 70)
print("SUCCESS! Contriever passage embedding generation complete!")
print(f"ZIP Archive:      {ZIP_DEST} ({zip_size_mb:.2f} MB)")
print(f"Passages:         {TOTAL_PASSAGES:,}")
print(f"Embedding Matrix: ({TOTAL_PASSAGES}, {EMBEDDING_DIM}) float32")
print("=" * 70)
print("\nNext Steps for Local Setup:")
print("1. Open your Google Drive and download:")
print(f"   {ZIP_DEST}")
print("2. Extract the three files into your local Nool project at:")
print("   indexes/contriever_hotpotqa_sample/")
print("3. Run the local verification script in your local terminal:")
print("   python scripts/verify_contriever_artifact.py")
print("4. Run a test dense search:")
print('   python scripts/search_contriever.py --query "Where was Marie Curie born?"')
print("=" * 70)